# Transfer Learning (Lab 6)

### Import Libraries

In [2]:
import tensorflow as tf
import matplotlib.pyplot as plt
import os
import numpy as np
import time

from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import load_img, img_to_array
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

### Data Preparation

1. Load Dataset

In [4]:
# Train
print("Train Dataset:")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    "train",
    image_size=(150, 150),
    validation_split=0.2,
    subset="training",
    seed=42
)

# Validate
print("\nValidate Dataset:")
validation_dataset = tf.keras.utils.image_dataset_from_directory(
    "train",
    image_size=(150,150),
    batch_size=32,
    validation_split=0.2,
    subset="validation",
    seed=42
)

class_names = train_dataset.class_names
num_classes = len(class_names)

Train Dataset:
Found 5004 files belonging to 10 classes.
Using 4004 files for training.

Validate Dataset:
Found 5004 files belonging to 10 classes.
Using 1000 files for validation.


2. Preprocess

- **preprocess_input:** Converts RGB channel order to BGR and Subtracts ImageNet's channel-specific mean pixel values
- **num_parallel_calls=tf.data.AUTOTUNE:** Lets TensorFlow automatically choose how much parallel processing to use for the mapping operation.
- **.prefetch(tf.data.AUTOTUNE):** Allows TensorFlow to prepare upcoming batches while the model trains on the current batch, helping reduce waiting time

In [5]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.1)
])

def prepare_train(images, labels):
    images = data_augmentation(images, training=True)
    images = preprocess_input(images)
    return images, labels

def prepare_validate(images, labels):
    images = preprocess_input(images)
    return images, labels

train_dataset = train_dataset.map(
    prepare_train,
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

validation_dataset = validation_dataset.map(
    prepare_validate,
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

### Train Models

1. Model 1: Feature Extraction only

In [4]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(150, 150, 3)
)

# Freeze all layers
base_model.trainable = False

# Build the model
inputs = Input(shape=(150, 150, 3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

x = Dense(256,activation="relu",kernel_regularizer=tf.keras.regularizers.l2(0.0001))(x)

x = Dropout(0.5)(x)

outputs = Dense(num_classes, activation="softmax")(x)

# Compile the model
model_feature = tf.keras.Model(inputs, outputs)
model_feature.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# Early stopping
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# Reduce learning rate (if val_loss reduce)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

# Train the model
start_time = time.perf_counter()
history_feature = model_feature.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=25,
    callbacks=[early_stopping, reduce_lr]
)
feature_time = time.perf_counter() - start_time

# Evaluate the model
loss, accuracy = model_feature.evaluate(
    validation_dataset,
    verbose=1
)

# Save the model
model_feature.save("resnet50_feature_extraction.keras")

# Display results
print(f"Training time: {feature_time:.2f} seconds")
print(f"Final validation accuracy: {accuracy * 100:.2f}%")
print(f"Best validation accuracy during training: {max(history_feature.history['val_accuracy']) * 100:.2f}")

Epoch 1/25


c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


126/126 ━━━━━━━━━━━━━━━━━━━━ 81s 600ms/step - accuracy: 0.5090 - loss: 1.5166 - val_accuracy: 0.6350 - val_loss: 1.1612 - learning_rate: 3.0000e-04
Epoch 2/25
126/126 ━━━━━━━━━━━━━━━━━━━━ 68s 533ms/step - accuracy: 0.6444 - loss: 1.0890 - val_accuracy: 0.6400 - val_loss: 1.1251 - learning_rate: 3.0000e-04
Epoch 3/25
126/126 ━━━━━━━━━━━━━━━━━━━━ 66s 522ms/step - accuracy: 0.6961 - loss: 0.9323 - val_accuracy: 0.6510 - val_loss: 1.0879 - learning_rate: 3.0000e-04
Epoch 4/25
126/126 ━━━━━━━━━━━━━━━━━━━━ 68s 531ms/step - accuracy: 0.7385 - loss: 0.8215 - val_accuracy: 0.6550 - val_loss: 1.1412 - learning_rate: 3.0000e-04
Epoch 5/25
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 417ms/step - accuracy: 0.7630 - loss: 0.7354
Epoch 5: ReduceLROnPlateau reducing learning rate to 6.000000284984708e-05.
126/126 ━━━━━━━━━━━━━━━━━━━━ 66s 521ms/step - accuracy: 0.7630 - loss: 0.7354 - val_accuracy: 0.6530 - val_loss: 1.1752 - learning_rate: 3.0000e-04
Epoch 6/25
126/126 ━━━━━━━━━━━━━━━━━━━━ 72s 563ms/step - accura

2. Model 2: Fine-tune (Unfreeze 30 top layers)

In [8]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(150, 150, 3)
)

base_model.trainable = False # Freeze all ResNet50 layers initially (unfreeze later)

# Build the model
inputs = Input(shape=(150, 150, 3))

x = base_model(inputs, training=False)

x = GlobalAveragePooling2D()(x)

x = Dense(256,activation="relu",kernel_regularizer=tf.keras.regularizers.l2(0.0001))(x)

x = Dropout(0.5)(x)

outputs = Dense(num_classes, activation="softmax")(x)

# Compile the model
model_ft30 = tf.keras.Model(inputs, outputs)
model_ft30.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Early stopping
early_stopping_head = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

# Reduce learning rate (if val_loss reduce)
    # ReduceLROnPlateau: Reduces the learning rate when progress stalls
    # factor=0.2: Multiplies the current learning rate by 0.2 when triggered, reducing it to 20% of its previous value
    # patience=2: Waits for 2 consecutive epochs without improvement before reducing the learning rate.
    # min_lr=1e-7: Sets the minimum learning rate to \(0.0000001\). It will not decrease below this value.
reduce_lr_head = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

start_time = time.perf_counter()

# Train model (with feature extraction only, no fine-tuning yet)
history_head = model_ft30.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=15,
    callbacks=[early_stopping_head, reduce_lr_head]
)

# Fine-tune last 30 layers (30 top layers)
base_model.trainable = True

freeze_until = len(base_model.layers) - 30

for i, layer in enumerate(base_model.layers):
    layer.trainable = i >= freeze_until

    # Keep Batch Normalization layers frozen
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

# Recompile after changing trainable layers
model_ft30.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stopping_ft = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr_ft = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=2,
    min_lr=1e-7,
    verbose=1
)

# Train model (with both feature extraction and fine-tuning)
history_ft30 = model_ft30.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=20,
    callbacks=[early_stopping_ft, reduce_lr_ft]
)

ft30_time = time.perf_counter() - start_time

# Evaluate
loss, accuracy = model_ft30.evaluate(
    validation_dataset,
    verbose=1
)

# Save
model_ft30.save("resnet50_finetune_30.keras")

# Display results
print(f"\nTotal training time: {ft30_time:.2f} seconds")
print(f"Final validation accuracy: {accuracy * 100:.2f}%")
print(f"Peak validation accuracy during fine-tuning: {max(history_ft30.history['val_accuracy']) * 100:.2f}")

Phase 1: Feature Extraction
Epoch 1/15


c:\Users\Lenovo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


126/126 ━━━━━━━━━━━━━━━━━━━━ 80s 595ms/step - accuracy: 0.4883 - loss: 1.5636 - val_accuracy: 0.6080 - val_loss: 1.2206 - learning_rate: 3.0000e-04
Epoch 2/15
126/126 ━━━━━━━━━━━━━━━━━━━━ 67s 530ms/step - accuracy: 0.6484 - loss: 1.0820 - val_accuracy: 0.6400 - val_loss: 1.1654 - learning_rate: 3.0000e-04
Epoch 3/15
126/126 ━━━━━━━━━━━━━━━━━━━━ 65s 512ms/step - accuracy: 0.6951 - loss: 0.9360 - val_accuracy: 0.6400 - val_loss: 1.1637 - learning_rate: 3.0000e-04
Epoch 4/15
126/126 ━━━━━━━━━━━━━━━━━━━━ 66s 518ms/step - accuracy: 0.7340 - loss: 0.8297 - val_accuracy: 0.6440 - val_loss: 1.1656 - learning_rate: 3.0000e-04
Epoch 5/15
126/126 ━━━━━━━━━━━━━━━━━━━━ 67s 526ms/step - accuracy: 0.7617 - loss: 0.7427 - val_accuracy: 0.6530 - val_loss: 1.1586 - learning_rate: 3.0000e-04
Epoch 6/15
126/126 ━━━━━━━━━━━━━━━━━━━━ 65s 511ms/step - accuracy: 0.7870 - loss: 0.6800 - val_accuracy: 0.6550 - val_loss: 1.1811 - learning_rate: 3.0000e-04
Epoch 7/15
126/126 ━━━━━━━━━━━━━━━━━━━━ 0s 413ms/step - a